## До того как вы приступите к решению:
**Tools → Settings → Editor → completions / suggestions / linting → disable**

## Задание 1

Допишите 2 реализации функции `increment()`, которая **увеличивает глобальную переменную `counter` на 1**:

**с/без** (!) python синтаксического сахара. Сигнатуру функции менять нельзя.

**1.1: с python синтаксическим сахаром**

In [1]:
counter = 0

def increment():
  global counter
  counter += 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')

counter=2 -- great!


**1.2: без python синтаксического сахара**

In [2]:
counter = 0

def increment():
  globals()['counter'] += 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')

counter=2 -- great!


## Задание 2

Достаньте **только функцию `sqrt`** из модуля `math` и исполните sqrt(169).  
Нельзя исполнять `import math`.

Подготовьте 2 решения.

...

In [3]:
# Решение 1: from ... import (синтаксический сахар)
from math import sqrt
result = sqrt(169)

assert result == 13

# Решение 2: через __import__ без синтаксического сахара
sqrt2 = __import__('math').sqrt
result = sqrt2(169)

assert result == 13

## Задание 3

Динамический импорт и перезагрузка.

1. Создайте модуль `mod.py`:

In [4]:
# 1. Создаём mod.py с msg = 'A'
with open('mod.py', 'w') as f:
    f.write('msg = "A"\n')

2. Импортируйте его и выведите `msg`

3. Измените `msg` на `B` в файле

4. Без перезагрузки сессии ноутбука, выведите новое значение `msg`

In [5]:
import importlib
import mod

# 2. Импортируем и выводим msg
print(mod.msg)  # A

# 3. Изменяем msg на 'B' в файле
with open('mod.py', 'w') as f:
    f.write('msg = "B"\n')

# 4. Перезагружаем модуль без перезапуска сессии
importlib.reload(mod)
print(mod.msg)  # B

A
B


## Задание 4

У вас есть дирректория `pkg`:

In [6]:
!mkdir -p pkg

In [7]:
%%writefile pkg/m1.py
pi = 3.1415_92_65
_e = 2.7
__i = -1

Overwriting pkg/m1.py


```
pkg/
└── m1.py
```

Ниже ячейки для вашего кода, а после задание

### **Первый способ**: через модуль из стандартной библиотеки CPython

In [8]:
# Способ 1: через importlib (модуль стандартной библиотеки CPython)
# Создаём pkg/__init__.py, который использует importlib для загрузки m1
# и выставляет публичные имена в пространство имён пакета
import importlib, os

init_code = '''\
import importlib, os, sys
_m1 = importlib.import_module(__name__ + '.m1')
globals().update({k: v for k, v in vars(_m1).items() if not k.startswith('_')})
'''

os.makedirs('pkg', exist_ok=True)
with open('pkg/__init__.py', 'w') as f:
    f.write(init_code)

### **Второй способ**: в одну строчку без доп.модулей

In [9]:
# Способ 2: в одну строчку без доп. модулей
# from .m1 import * — реэкспортирует все публичные имена из m1 (т.е. pi)
with open('pkg/__init__.py', 'w') as f:
    f.write('from .m1 import *\n')

### Текст задания:
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.  
Вам необходимо изменить структуру `pkg` пакета / содержимое его модулей, чтобы следующий код выполнялся корректно:

In [10]:
from pkg import *
pi

3.14159265

**Важно!**  
При обновлении любых данных в дирректории проекта, вам необходимо перезагружать сессию ipynb:  
`Runtime --> Restart Session` и перезапустить необходимые ячейки задания,  
иначе результаты могут быть для вас некорректными.

## Задание 5

При правильно решённом **задании 4** вам необходимо:
- изменить `pkg`
- дописать код ниже

так, чтобы "дотянуться" до `__i`.  
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.    
Если вы решите перезагрузить сессию, то для решения **задания 5** необходимо перезапустить ячейки **задания 4**.  

In [11]:
import sys

# Изменяем pkg/__init__.py:
# 1. from .m1 import * — экспортирует публичные имена (pi)
# 2. from .m1 import __i — явно импортируем __i (имя с __ не попадает в import *)
# 3. __all__ = ['pi', '__i'] — явно указываем, что from pkg import * должен экспортировать
with open('pkg/__init__.py', 'w') as f:
    f.write('from .m1 import *\n')
    f.write('from .m1 import __i\n')
    f.write("__all__ = ['pi', '__i']\n")

# Сбрасываем кэш модуля, чтобы изменения подхватились без перезапуска сессии
for key in list(sys.modules.keys()):
    if 'pkg' in key:
        del sys.modules[key]

### Решение

In [12]:
from pkg import *
__i

-1

## Задание 6

Изменяемое замыкание. Почему этот код ведёт себя неожиданно? Исправьте.

In [13]:
# Проблема: accs содержит только int-ы (0, 0, 0) — функции accumulator туда не добавляются.
# accs.append(0) добавляет int, а не функцию.
# Поэтому acc_list[0] == 0 (int), а не callable → TypeError.
#
# Исправление: хранить функции и значения раздельно.
# Каждый аккумулятор замыкается на свой изменяемый список [0],
# чтобы состояние было независимым для каждого экземпляра.

def create_accumulators():
    funcs = []
    for i in range(3):
        state = [0]  # изменяемый контейнер — отдельный для каждого аккумулятора
        def accumulator(x, s=state):
            s[0] += x
            return s[0]
        funcs.append(accumulator)
    return funcs

acc_list = create_accumulators()
print(acc_list[0](10))  # 10
print(acc_list[1](5))   # 5
print(acc_list[0](3))   # 13

10
5
13


## Задание 7
При перезапуске сессии ноутбука решение задачи начинается сначала

### 7.1: Востановите работу `print`, не используя `del`

In [14]:
print = 1

In [15]:
# 7.1: Восстановить print без del
# print = 1 создал имя 'print' в глобальном пространстве имён, перекрыв встроенный print.
# Без del — берём оригинальный print из модуля builtins напрямую.
import builtins
print = builtins.print
print('print restored!')  # проверка

print restored!


### 7.2: Удалите объект `print`, после востановите его функционал

In [16]:
# 7.2: Удалить print и восстановить
# Сначала перезаписываем, чтобы было что удалять (имитируем состояние после print = 1)
print = 1

# Удаляем перезаписанный print из глобального пространства имён
del print
# Теперь 'print' снова резолвится из builtins — встроенный print восстановлен автоматически
print('print restored after del!')  # проверка

# Альтернатива: явно восстановить через builtins
# import builtins; print = builtins.print

print restored after del!


## Задание 8

Замыкание с изменяемым состоянием. Создайте функцию-счётчик, которая запоминает количество вызовов между разными экземплярами:

In [17]:
# Общее состояние хранится в изменяемом контейнере _shared,
# который определён ВНЕ make_shared_counter — на уровне модуля.
# Все экземпляры (c1, c2, ...) замыкаются на один и тот же _shared.

_shared = [0]  # единое состояние для всех экземпляров

def make_shared_counter():
    def counter():
        _shared[0] += 1
        return _shared[0]
    return counter

c1 = make_shared_counter()
c2 = make_shared_counter()

print(c1())  # 1
print(c2())  # 2
print(c1())  # 3

1
2
3


## Задание 9

Допишите код, чтобы функция `outer` возвращала **словарь с тремя замыканиями**: `add()`, `mul()`, `get()` — работающими с одной и той же закрытой переменной `value`.

In [18]:
def outer(val=0):
    # value — int (immutable), поэтому используем изменяемый контейнер [val]
    value = [val]

    def add(x):
        value[0] += x

    def mul(x):
        value[0] *= x

    def get():
        return value[0]

    return {'add': add, 'mul': mul, 'get': get}

obj = outer(10)
obj['add'](5)       # value = 10 + 5 = 15
obj['mul'](2)       # value = 15 * 2 = 30
assert obj['get']() == 30
print(obj['get']())  # 30

30


## Задание 10

Создать closure, которая принимает функцию и возвращает новую функцию с кэшированием результатов (мемоизацией).

*Теоретическая справка:*

Функция `memoize` принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

В примере с функцией `fib` (числа Фибоначчи) мемоизация уменьшает количество рекурсивных вызовов с экспоненциального до линейного, так как результаты для каждого n вычисляются один раз.

Таким образом, мемоизация экономит время за счёт памяти — класическая оптимизация "время против памяти" — и особенно полезна для функций с дорогими вычислениями и повторяющимися входами.

Алгоритм для решения:

- Функция memoize принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

- Внутренняя функция wrapper проверяет, есть ли для данного входного аргумента (x) уже вычисленный результат в словаре cache.

- Если результат есть, то он возвращается из кеша, и вычисления не повторяются.

- Если нет, то вызывается исходная функция func(x), результат сохраняется в cache и возвращается.


In [19]:
def memoize(func):
    cache = {}  # замыкание: cache живёт в области видимости memoize
    def wrapper(x):
        if x not in cache:
            cache[x] = func(x)  # вычисляем и сохраняем результат
        return cache[x]         # возвращаем из кэша
    return wrapper

@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

print(fib(10))  # 55

55
